# Notebook 03 - Final Dataset Builder & Split Strategy

Notebook ini membangun dataset final dari output Notebook 02:

1. Membaca `review_level_relation_policy.csv`, `image_level_relation_policy.csv`, dan `aspect_level_relation_matrix.csv`.
2. Menghapus data artifact yang tidak layak training, terutama `#NAME?` dan `manual_review_or_exclude`.
3. Membuat split `train`, `val`, dan `test` di level `ID_Review`, sehingga semua gambar dari review yang sama tidak bocor ke split berbeda.
4. Menyimpan beberapa varian dataset:
   - `core_clean_training`
   - `relation_aware_training`
   - `primary_training`
   - `contradiction_ablation`
   - `all_none_control`
5. Membuat manifest image dan aspect-level long format untuk training/evaluasi MABSA.

Notebook ini didesain untuk Kaggle. Output utama akan tersimpan di:

`/kaggle/working/mabsa_final_dataset_outputs`

## Dasar Metodologi

Notebook ini mengikuti tiga prinsip metodologis:

1. **Aspect-level target, bukan sentiment umum.** SemEval-2014 Task 4 mendefinisikan ABSA sebagai prediksi sentimen terhadap aspek tertentu. Karena itu, label per aspek tetap dipertahankan sebagai 7 target terpisah: Kamar, Kebersihan, Pelayanan, Harga, Lokasi, Fasilitas, dan Makanan.

2. **Multimodal tidak berarti semua aspek harus muncul di dua modality.** MIMN memperkenalkan MABSA sebagai tugas yang memodelkan interaksi aspek, teks, dan gambar. Jika aspek hanya muncul di teks atau hanya terlihat pada gambar, data tersebut tidak otomatis salah. Yang penting adalah relasinya ditandai.

3. **Noisy atau text-unrelated images harus dikendalikan.** M2DF menunjukkan bahwa banyak gambar dalam dataset MABSA dapat tidak berhubungan dengan teks dan bisa mengganggu learning. Karena itu, Notebook 3 tidak mencampur semua data secara naif, tetapi memisahkan clean data, relation-aware data, contradiction data, dan artifact data.

Referensi:

- Pontiki et al. (2014), SemEval-2014 Task 4: Aspect Based Sentiment Analysis. https://aclanthology.org/S14-2004/
- Xu, Mao, & Chen (2019), Multi-Interactive Memory Network for Aspect Based Multimodal Sentiment Analysis. https://ojs.aaai.org/index.php/AAAI/article/view/3807
- Zhao et al. (2023), M2DF: Multi-grained Multi-curriculum Denoising Framework for Multimodal Aspect-based Sentiment Analysis. https://aclanthology.org/2023.emnlp-main.561/

### 1. Import Library dan Persiapan Direktori Output


In [1]:
from pathlib import Path
import json
import os
import re
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

SEED = 42
ASPECTS = ["Kamar", "Kebersihan", "Pelayanan", "Harga", "Lokasi", "Fasilitas", "Makanan"]
VALID_LABELS = ["None", "Negatif", "Netral", "Positif"]
LABEL_TO_ID = {"None": 0, "Negatif": 1, "Netral": 2, "Positif": 3}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
ASPECT_TO_ID = {aspect: idx for idx, aspect in enumerate(ASPECTS)}

IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_DIR = Path("/kaggle/working/mabsa_final_dataset_outputs") if IS_KAGGLE else Path("mabsa_final_dataset_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Kaggle environment:", IS_KAGGLE)

Output directory: /kaggle/working/mabsa_final_dataset_outputs
Kaggle environment: True


### 2. Fungsi Pencarian Path File Output dari Notebook 02


In [2]:
def find_file(filename):
    candidates = []

    if IS_KAGGLE:
        candidates.append(Path("/kaggle/working/mabsa_relation_policy_outputs") / filename)
        input_root = Path("/kaggle/input")
        if input_root.exists():
            candidates.extend(input_root.rglob(filename))

    local_candidates = [
        Path(r"C:\Users\cencen04_\Documents\Codex\2026-05-23\files-mentioned-by-the-user-data\mabsa_relation_policy_outputs") / filename,
        Path(r"G:\Skripsi\Data Preprocessing\mabsa_relation_policy_outputs") / filename,
    ]
    candidates.extend(local_candidates)

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Tidak menemukan {filename}. Jalankan Notebook 02 lebih dulu atau upload output Notebook 02 sebagai Kaggle dataset."
    )


REVIEW_POLICY_PATH = find_file("review_level_relation_policy.csv")
IMAGE_POLICY_PATH = find_file("image_level_relation_policy.csv")
ASPECT_MATRIX_PATH = find_file("aspect_level_relation_matrix.csv")

print("Review policy:", REVIEW_POLICY_PATH)
print("Image policy :", IMAGE_POLICY_PATH)
print("Aspect matrix:", ASPECT_MATRIX_PATH)

Review policy: /kaggle/input/datasets/vince0014/datasetinput-notebook3/review_level_relation_policy.csv
Image policy : /kaggle/input/datasets/vince0014/datasetinput-notebook3/image_level_relation_policy.csv
Aspect matrix: /kaggle/input/datasets/vince0014/datasetinput-notebook3/aspect_level_relation_matrix.csv


### 3. Memuat Data Hasil Ekspor Taxonomy dan Policy


In [3]:
review_policy = pd.read_csv(REVIEW_POLICY_PATH)
image_policy = pd.read_csv(IMAGE_POLICY_PATH)
aspect_matrix = pd.read_csv(ASPECT_MATRIX_PATH)

print("review_policy shape:", review_policy.shape)
print("image_policy shape :", image_policy.shape)
print("aspect_matrix shape:", aspect_matrix.shape)

display(review_policy.head(3))
display(image_policy.head(3))
display(aspect_matrix.head(3))

review_policy shape: (8038, 41)
image_policy shape : (17385, 24)
aspect_matrix shape: (56266, 12)


,ID_Review,Platform,Wilayah,Nama_Hotel,Review_Date,Text_Review,image_count,image_files_sample,text_Kamar,text_Kebersihan,text_Pelayanan,text_Harga,text_Lokasi,text_Fasilitas,text_Makanan,image_Kamar,image_Kebersihan,image_Pelayanan,image_Harga,image_Lokasi,image_Fasilitas,image_Makanan,text_aspects,image_aspects,shared_aspects,hard_contradiction_aspects,soft_disagreement_aspects,text_aspect_count,image_aspect_count,shared_aspect_count,hard_contradiction_count,soft_disagreement_count,relation_overlap_ratio,relation_category,relation_strength,policy_bucket,recommended_fusion_strategy,use_for_primary_training,use_for_ablation,needs_manual_review,text_artifact_issue
0,1,Traveloka,Bandung,Atlantic City Hotel,5/4/2026,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,Positif,Positif,Positif,NaN,Positif,Negatif,Netral,NaN,Positif,NaN,NaN,NaN,Positif,NaN,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kebersihan,Fasilitas|Kebersihan,Fasilitas,NaN,6,2,2,1,0,0.3333,correlated_but_contradictive,conflict,contradiction_ablation,contradiction_aware_or_gated_fusion,False,True,True,False
1,10,Traveloka,Bandung,Atlantic City Hotel,1/17/2026,"Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher makan malam Natal, terima kasih.",2,R10_G1_8debfff9.jpg|R10_G2_b29ac5d6.jpg,NaN,NaN,Positif,NaN,NaN,NaN,NaN,NaN,Positif,NaN,NaN,Positif,NaN,Positif,Pelayanan,Kebersihan|Lokasi|Makanan,NaN,NaN,NaN,1,3,0,0,0,0.0000,uncorrelated_different_aspects,low,relation_aware_training,relevance_aware_gated_fusion,True,True,False,False
2,100,Traveloka,Bandung,Atlantic City Hotel,9/24/2022,"Lokasi strategis. Petugas ramah. Sekuriti top abis, ramah dan pintar memarkirkan mobil, walaupun sempit parkiran di basement, jadi mudah. Makanan rating 7/1...",9,R100_G1_3131b00a.jpg|R100_G2_a4e9a23b.jpg|R100_G3_4541d369.jpg|R100_G4_70815ec9.jpg|R100_G5_898c941e.jpg|R100_G6_57cc0cdf.jpg|R100_G7_5a2d3bd7.jpg|R100_G8_e...,Netral,Positif,Positif,Positif,Positif,NaN,Netral,Positif,Netral|Positif,Positif,NaN,Netral|Positif,Positif,NaN,Harga|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kamar|Kebersihan|Lokasi|Pelayanan,Kamar|Kebersihan|Lokasi|Pelayanan,NaN,Kamar,6,5,4,0,1,0.5714,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False


,ID_Review,Review_Date,Gambar_Ke,Image_FileName,Kamar,Kebersihan,Pelayanan,Harga,Lokasi,Fasilitas,Makanan,relation_category,relation_strength,policy_bucket,recommended_fusion_strategy,use_for_primary_training,use_for_ablation,needs_manual_review,text_artifact_issue,text_aspects,image_aspects,shared_aspects,hard_contradiction_aspects,soft_disagreement_aspects
0,1,5/4/2026,1,R1_G1_687bd728.jpg,NaN,Positif,NaN,NaN,NaN,Positif,NaN,correlated_but_contradictive,conflict,contradiction_ablation,contradiction_aware_or_gated_fusion,False,True,True,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kebersihan,Fasilitas|Kebersihan,Fasilitas,NaN
1,2,4/18/2026,1,R2_G1_eb4fea87.jpg,NaN,NaN,NaN,NaN,Positif,NaN,NaN,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Kamar|Kebersihan|Lokasi,Kamar|Kebersihan|Lokasi,NaN,NaN
2,2,4/18/2026,2,R2_G2_3b6a9a2a.jpg,Netral,Netral,NaN,NaN,NaN,NaN,NaN,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Kamar|Kebersihan|Lokasi,Kamar|Kebersihan|Lokasi,NaN,NaN


,ID_Review,Review_Date,Platform,Wilayah,Nama_Hotel,aspect,text_label,image_labels,aspect_relation_status,has_text_aspect,has_image_aspect,is_hard_contradiction
0,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Kamar,Positif,NaN,text_only,True,False,False
1,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Kebersihan,Positif,Positif,same_label,True,True,False
2,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Pelayanan,Positif,NaN,text_only,True,False,False


### 4. Normalisasi dan Konversi ID Label Numerik (Positif/Negatif/Netral/None)


In [4]:
def normalize_single_label(value):
    if pd.isna(value):
        return "None"
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return "None"
    return text


def normalize_agg_label(value):
    if pd.isna(value):
        return "None"
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return "None"
    parts = [normalize_single_label(part) for part in text.split("|")]
    parts = [part for part in parts if part != "None"]
    if not parts:
        return "None"
    unique_parts = sorted(set(parts), key=lambda item: VALID_LABELS.index(item) if item in VALID_LABELS else 999)
    return "|".join(unique_parts)


def image_label_mode(value):
    label_set = normalize_agg_label(value)
    if label_set == "None":
        return "None"
    parts = label_set.split("|")
    if len(parts) == 1:
        return parts[0]
    return "Mixed"


def label_id(value):
    value = normalize_single_label(value)
    return LABEL_TO_ID.get(value, -1)


def clean_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


for aspect in ASPECTS:
    review_policy[f"text_{aspect}"] = review_policy[f"text_{aspect}"].map(normalize_single_label)
    review_policy[f"image_{aspect}"] = review_policy[f"image_{aspect}"].map(normalize_agg_label)
    image_policy[aspect] = image_policy[aspect].map(normalize_single_label)

aspect_matrix["text_label"] = aspect_matrix["text_label"].map(normalize_single_label)
aspect_matrix["image_labels"] = aspect_matrix["image_labels"].map(normalize_agg_label)

for col in ["use_for_primary_training", "use_for_ablation", "needs_manual_review", "text_artifact_issue"]:
    review_policy[col] = review_policy[col].map(clean_bool)
    image_policy[col] = image_policy[col].map(clean_bool)

invalid_cells = []
for aspect in ASPECTS:
    for row in review_policy.loc[~review_policy[f"text_{aspect}"].isin(VALID_LABELS), ["ID_Review", f"text_{aspect}"]].itertuples(index=False):
        invalid_cells.append({"dataset": "review_text", "ID_Review": row[0], "aspect": aspect, "label": row[1]})
    for row in image_policy.loc[~image_policy[aspect].isin(VALID_LABELS), ["ID_Review", "Image_FileName", aspect]].itertuples(index=False):
        invalid_cells.append({"dataset": "image", "ID_Review": row[0], "Image_FileName": row[1], "aspect": aspect, "label": row[2]})

invalid_cells = pd.DataFrame(invalid_cells)
print("Invalid normalized label cells:", len(invalid_cells))
display(invalid_cells.head(20))

Invalid normalized label cells: 0


""


## Final Inclusion Policy

Keputusan final:

- `manual_review_or_exclude` dan `text_artifact_issue=True` dikeluarkan dari dataset training/evaluasi utama.
- `core_clean_training` dipakai sebagai baseline multimodal paling aman.
- `relation_aware_training` tetap dipakai untuk eksperimen model yang memakai relevance/gating strategy.
- `contradiction_ablation` dipisahkan sebagai stress-test dan error analysis, bukan dicampur ke clean baseline.
- `all_none_control` dipertahankan sebagai kontrol negatif semua aspek `None`, tetapi diberi flag supaya mudah dievaluasi terpisah.

Label utama untuk training text/ABSA tetap berasal dari anotasi teks. Label image disimpan sebagai auxiliary visual evidence, bukan dipakai untuk menimpa label teks. Ini penting agar kontradiksi antar modality tidak dipalsukan menjadi satu label tunggal.

### 5. Penentuan Final Exclusions dan Penugasan Peran Dataset (Dataset Role)


In [5]:
final_review = review_policy.copy()

final_review["is_excluded_final"] = (
    final_review["text_artifact_issue"]
    | final_review["policy_bucket"].eq("manual_review_or_exclude")
)

conditions = [
    final_review["is_excluded_final"],
    final_review["policy_bucket"].eq("core_clean_training"),
    final_review["policy_bucket"].eq("relation_aware_training"),
    final_review["policy_bucket"].eq("contradiction_ablation"),
    final_review["policy_bucket"].eq("all_none_control"),
]
choices = [
    "excluded",
    "core_clean_training",
    "relation_aware_training",
    "contradiction_ablation",
    "all_none_control",
]
final_review["dataset_role"] = np.select(conditions, choices, default="other_review_needed")

final_review["primary_training_candidate"] = (
    final_review["dataset_role"].isin(["core_clean_training", "relation_aware_training", "all_none_control"])
)
final_review["core_clean_candidate"] = final_review["dataset_role"].eq("core_clean_training")
final_review["relation_aware_candidate"] = final_review["dataset_role"].isin(["core_clean_training", "relation_aware_training"])
final_review["contradiction_ablation_candidate"] = final_review["dataset_role"].eq("contradiction_ablation")

for aspect in ASPECTS:
    final_review[f"label_text_{aspect}"] = final_review[f"text_{aspect}"].map(normalize_single_label)
    final_review[f"label_image_agg_{aspect}"] = final_review[f"image_{aspect}"].map(normalize_agg_label)
    final_review[f"label_image_mode_{aspect}"] = final_review[f"image_{aspect}"].map(image_label_mode)
    final_review[f"label_text_id_{aspect}"] = final_review[f"label_text_{aspect}"].map(label_id)
    final_review[f"label_image_mode_id_{aspect}"] = final_review[f"label_image_mode_{aspect}"].map(lambda x: LABEL_TO_ID.get(x, -1))

role_counts = (
    final_review["dataset_role"]
    .value_counts()
    .rename_axis("dataset_role")
    .reset_index(name="count")
)
role_counts["percentage"] = (role_counts["count"] / len(final_review) * 100).round(2)

display(role_counts)
display(final_review[["ID_Review", "dataset_role", "policy_bucket", "relation_category", "Text_Review"]].head(10))

,dataset_role,count,percentage
0,core_clean_training,4628,57.58
1,relation_aware_training,2209,27.48
2,contradiction_ablation,1176,14.63
3,all_none_control,17,0.21
4,excluded,8,0.10


,ID_Review,dataset_role,policy_bucket,relation_category,Text_Review
0,1,contradiction_ablation,contradiction_ablation,correlated_but_contradictive,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist..."
1,10,relation_aware_training,relation_aware_training,uncorrelated_different_aspects,"Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher makan malam Natal, terima kasih."
2,100,core_clean_training,core_clean_training,correlated_non_contradictive,"Lokasi strategis. Petugas ramah. Sekuriti top abis, ramah dan pintar memarkirkan mobil, walaupun sempit parkiran di basement, jadi mudah. Makanan rating 7/1..."
3,1000,contradiction_ablation,contradiction_ablation,correlated_but_contradictive,"sebuah hotel yang sangat unik, dengan tema kuning, kamar, makanan, hiburan yang luar biasa. Keluarga saya sangat puas dengan semua layanan, keramahtamahan, ..."
4,1001,core_clean_training,core_clean_training,correlated_non_contradictive,Semuanya baik-baik saja! Satu-satunya hal yang tidak memuaskan hanya tentang spa. Saya ingin mencobanya. Jadi saya menelepon nomor itu dan meminta perawatan...
5,1002,core_clean_training,core_clean_training,correlated_non_contradictive,"staf yang baik dan ramah. Proses check-in dan checkout yang mudah. Lokasi sangat bagus, terhubung dengan Paskal 23 mal. Suasana yang nyaman di lobi dan kama..."
6,1003,core_clean_training,core_clean_training,correlated_non_contradictive,"Sangat cocok dengan keluarga, dekat dengan Mal, enak banget."
7,1004,core_clean_training,core_clean_training,correlated_non_contradictive,Memuaskan! Awalnya sempat agak ragu juga karena liat beberapa review yang ga bagus tentang Yello. Tapi semua ga terbukti pas aku sama keluarga menginap 1 ma...
8,1005,core_clean_training,core_clean_training,correlated_non_contradictive,"Hotel bagus, pelayanan prima."
9,1006,core_clean_training,core_clean_training,correlated_non_contradictive,"Buat yang mencari non stop hiburan YELLO Hotel ini sangat cocok. Setiap sudut begitu artsy dan instagramable. Fasilitas lengkap, ratenya terjangkau. Yang pa..."


### 6. Pembagian Data (Train/Val/Test Split) dengan Stratifikasi Multi-kriteria


In [6]:
def assign_review_level_split(df, strata_col, seed=42, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratio harus berjumlah 1.0")

    rng = np.random.default_rng(seed)
    split = pd.Series(index=df.index, dtype="object")

    for _, group in df.groupby(strata_col, dropna=False):
        idx = group.index.to_numpy(copy=True)
        rng.shuffle(idx)
        n = len(idx)

        if n == 1:
            n_test = 0
            n_val = 0
        elif n == 2:
            n_test = 1
            n_val = 0
        elif n == 3:
            n_test = 1
            n_val = 1
        else:
            n_test = max(1, int(round(n * test_ratio)))
            n_val = max(1, int(round(n * val_ratio)))
            if n_test + n_val >= n:
                n_test = max(1, min(n_test, n - 2))
                n_val = max(1, min(n_val, n - n_test - 1))

        test_idx = idx[:n_test]
        val_idx = idx[n_test:n_test + n_val]
        train_idx = idx[n_test + n_val:]

        split.loc[test_idx] = "test"
        split.loc[val_idx] = "val"
        split.loc[train_idx] = "train"

    return split


valid_mask = ~final_review["is_excluded_final"]
final_review["split_strata"] = final_review["dataset_role"] + "__" + final_review["relation_category"].astype(str)
final_review["split"] = "excluded"
final_review.loc[valid_mask, "split"] = assign_review_level_split(
    final_review.loc[valid_mask],
    strata_col="split_strata",
    seed=SEED,
)

split_overview = (
    final_review
    .groupby(["dataset_role", "split"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_role", "split"])
)
split_overview["percentage_of_role"] = (
    split_overview["count"]
    / split_overview.groupby("dataset_role")["count"].transform("sum")
    * 100
).round(2)

display(split_overview)

,dataset_role,split,count,percentage_of_role
0,all_none_control,test,3,17.65
1,all_none_control,train,11,64.71
2,all_none_control,val,3,17.65
3,contradiction_ablation,test,176,14.97
4,contradiction_ablation,train,824,70.07
5,contradiction_ablation,val,176,14.97
6,core_clean_training,test,694,15.00
7,core_clean_training,train,3240,70.01
8,core_clean_training,val,694,15.00
9,excluded,excluded,8,100.00


### 7. Resolusi dan Pengecekan Lokasi Fisik File Gambar (Image Manifest)


In [7]:
def explode_image_file_sample(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return []
    return [item.strip() for item in text.split("|") if item.strip()]


def build_image_path_index(target_filenames):
    target_filenames = set(target_filenames)
    image_index = {}
    if not target_filenames:
        return image_index

    roots = []
    if IS_KAGGLE and Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))

    local_roots = [
        Path(r"G:\Skripsi"),
        Path(r"C:\Users\cencen04_\Downloads"),
    ]
    roots.extend([root for root in local_roots if root.exists()])

    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

    for root in roots:
        try:
            for path in root.rglob("*"):
                if len(image_index) == len(target_filenames):
                    return image_index
                if path.is_file() and path.suffix.lower() in image_extensions and path.name in target_filenames:
                    image_index[path.name] = str(path)
        except Exception as exc:
            print(f"Skip image scan root {root}: {exc}")

    return image_index


image_manifest = image_policy.merge(
    final_review[[
        "ID_Review", "Platform", "Wilayah", "Nama_Hotel", "Text_Review",
        "dataset_role", "split", "is_excluded_final",
        "relation_category", "relation_strength", "policy_bucket",
        "recommended_fusion_strategy", "primary_training_candidate",
        "core_clean_candidate", "relation_aware_candidate", "contradiction_ablation_candidate",
    ]],
    on="ID_Review",
    how="left",
    suffixes=("", "_review"),
)

target_filenames = image_manifest.loc[~image_manifest["is_excluded_final"], "Image_FileName"].dropna().astype(str).unique()
print("Scanning image paths for", len(target_filenames), "unique filenames...")
image_path_index = build_image_path_index(target_filenames)
print("Found image paths:", len(image_path_index))

image_manifest["Image_Path"] = image_manifest["Image_FileName"].map(image_path_index).fillna("")
image_manifest["image_path_found"] = image_manifest["Image_Path"].ne("")

for aspect in ASPECTS:
    image_manifest[f"label_image_{aspect}"] = image_manifest[aspect].map(normalize_single_label)
    image_manifest[f"label_image_id_{aspect}"] = image_manifest[f"label_image_{aspect}"].map(label_id)

display(image_manifest.head(10))
print("Image path found ratio:", round(image_manifest["image_path_found"].mean() * 100, 2), "%")

Scanning image paths for 17371 unique filenames...
Found image paths: 17371


,ID_Review,Review_Date,Gambar_Ke,Image_FileName,Kamar,Kebersihan,Pelayanan,Harga,Lokasi,Fasilitas,Makanan,relation_category,relation_strength,policy_bucket,recommended_fusion_strategy,use_for_primary_training,use_for_ablation,needs_manual_review,text_artifact_issue,text_aspects,image_aspects,shared_aspects,hard_contradiction_aspects,soft_disagreement_aspects,Platform,Wilayah,Nama_Hotel,Text_Review,dataset_role,split,is_excluded_final,relation_category_review,relation_strength_review,policy_bucket_review,recommended_fusion_strategy_review,primary_training_candidate,core_clean_candidate,relation_aware_candidate,contradiction_ablation_candidate,Image_Path,image_path_found,label_image_Kamar,label_image_id_Kamar,label_image_Kebersihan,label_image_id_Kebersihan,label_image_Pelayanan,label_image_id_Pelayanan,label_image_Harga,label_image_id_Harga,label_image_Lokasi,label_image_id_Lokasi,label_image_Fasilitas,label_image_id_Fasilitas,label_image_Makanan,label_image_id_Makanan
0,1,5/4/2026,1,R1_G1_687bd728.jpg,None,Positif,None,None,None,Positif,None,correlated_but_contradictive,conflict,contradiction_ablation,contradiction_aware_or_gated_fusion,False,True,True,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kebersihan,Fasilitas|Kebersihan,Fasilitas,NaN,Traveloka,Bandung,Atlantic City Hotel,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",contradiction_ablation,train,False,correlated_but_contradictive,conflict,contradiction_ablation,contradiction_aware_or_gated_fusion,False,False,False,True,/kaggle/input/datasets/vince0014/image-cache/image_cache/R1_G1_687bd728.jpg,True,None,0,Positif,3,None,0,None,0,None,0,Positif,3,None,0
1,2,4/18/2026,1,R2_G1_eb4fea87.jpg,None,None,None,None,Positif,None,None,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Kamar|Kebersihan|Lokasi,Kamar|Kebersihan|Lokasi,NaN,NaN,Traveloka,Bandung,Atlantic City Hotel,"Menyenangkan menginap di Hotel Atlantic. Hotel berada di lokasi strategis dekat dengan Istana Plaza, halte transportasi umum, tempat kulineran, dan akses pe...",core_clean_training,val,False,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,True,True,False,/kaggle/input/datasets/vince0014/image-cache/image_cache/R2_G1_eb4fea87.jpg,True,None,0,None,0,None,0,None,0,Positif,3,None,0,None,0
2,2,4/18/2026,2,R2_G2_3b6a9a2a.jpg,Netral,Netral,None,None,None,None,None,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Kamar|Kebersihan|Lokasi,Kamar|Kebersihan|Lokasi,NaN,NaN,Traveloka,Bandung,Atlantic City Hotel,"Menyenangkan menginap di Hotel Atlantic. Hotel berada di lokasi strategis dekat dengan Istana Plaza, halte transportasi umum, tempat kulineran, dan akses pe...",core_clean_training,val,False,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,True,True,False,/kaggle/input/datasets/vince0014/image-cache/image_cache/R2_G2_3b6a9a2a.jpg,True,Netral,2,Netral,2,None,0,None,0,None,0,None,0,None,0
3,2,4/18/2026,3,R2_G3_ecd0ece1.jpg,Positif,Positif,None,None,None,None,None,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,False,False,False,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Kamar|Kebersihan|Lokasi,Kamar|Kebersihan|Lokasi,NaN,NaN,Traveloka,Bandung,Atlantic City Hotel,"Menyenangkan menginap di Hotel Atlantic. Hotel berada di lokasi strategis dekat dengan Istana Plaza, halte transportasi umum, tempat kulineran, dan akses pe...",core_clean_training,val,False,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,True,True,True,False,/kaggle/input/datasets/vince0014/image-cache/image_cache/R2_G3_ecd0ece1.jpg,True,Positif,3,Positif,3,N

Image path found ratio: 99.92 %


### 8. Penggabungan Data Aspek dengan Metadata Final (Aspect-Level Matrix)


In [8]:
aspect_long = aspect_matrix.merge(
    final_review[[
        "ID_Review", "Text_Review", "image_count", "image_files_sample",
        "dataset_role", "split", "is_excluded_final",
        "policy_bucket", "relation_category", "relation_strength",
        "recommended_fusion_strategy", "primary_training_candidate",
        "core_clean_candidate", "relation_aware_candidate", "contradiction_ablation_candidate",
    ]],
    on="ID_Review",
    how="left",
    suffixes=("", "_review"),
)

aspect_long["text_label"] = aspect_long["text_label"].map(normalize_single_label)
aspect_long["image_labels"] = aspect_long["image_labels"].map(normalize_agg_label)
aspect_long["image_label_mode"] = aspect_long["image_labels"].map(image_label_mode)
aspect_long["text_label_id"] = aspect_long["text_label"].map(label_id)
aspect_long["image_label_mode_id"] = aspect_long["image_label_mode"].map(lambda x: LABEL_TO_ID.get(x, -1))
aspect_long["aspect_id"] = aspect_long["aspect"].map(ASPECT_TO_ID)
aspect_long["is_text_target_positive_aspect"] = aspect_long["text_label"].ne("None")
aspect_long["is_image_target_positive_aspect"] = aspect_long["image_labels"].ne("None")

display(aspect_long.head(14))

,ID_Review,Review_Date,Platform,Wilayah,Nama_Hotel,aspect,text_label,image_labels,aspect_relation_status,has_text_aspect,has_image_aspect,is_hard_contradiction,Text_Review,image_count,image_files_sample,dataset_role,split,is_excluded_final,policy_bucket,relation_category,relation_strength,recommended_fusion_strategy,primary_training_candidate,core_clean_candidate,relation_aware_candidate,contradiction_ablation_candidate,image_label_mode,text_label_id,image_label_mode_id,aspect_id,is_text_target_positive_aspect,is_image_target_positive_aspect
0,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Kamar,Positif,None,text_only,True,False,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,None,3,0,0,True,False
1,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Kebersihan,Positif,Positif,same_label,True,True,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,Positif,3,3,1,True,True
2,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Pelayanan,Positif,None,text_only,True,False,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,None,3,0,2,True,False
3,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Harga,None,None,both_none,False,False,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,None,0,0,3,False,False
4,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Lokasi,Positif,None,text_only,True,False,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,None,3,0,4,True,False
5,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Fasilitas,Negatif,Positif,hard_contradiction,True,True,True,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,Positif,1,3,5,True,True
6,1,5/4/2026,Traveloka,Bandung,Atlantic City Hotel,Makanan,Netral,None,text_only,True,False,False,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat ist...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,False,contradiction_ablation,correlated_but_contradictive,conflict,contradiction_aware_or_gated_fusion,False,False,False,True,None,2,0,6,True,False
7,10,1/17/2026,Traveloka,Bandung,Atlantic City Hotel,Kamar,None,None,both_none,False,False,False,"Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher mak

### 9. Pembentukan Subset Dataset Spesifik (Primary, Core, Relation-Aware)


In [9]:
def subset_review(name, mask):
    df = final_review.loc[mask].copy()
    print(f"{name}: {df.shape}")
    return df


review_all_valid = subset_review("all_valid", ~final_review["is_excluded_final"])
review_primary = subset_review("primary_training", final_review["primary_training_candidate"] & ~final_review["is_excluded_final"])
review_core_clean = subset_review("core_clean", final_review["core_clean_candidate"] & ~final_review["is_excluded_final"])
review_relation_aware = subset_review("relation_aware", final_review["relation_aware_candidate"] & ~final_review["is_excluded_final"])
review_contradiction = subset_review("contradiction_ablation", final_review["contradiction_ablation_candidate"] & ~final_review["is_excluded_final"])
review_all_none = subset_review("all_none_control", final_review["dataset_role"].eq("all_none_control") & ~final_review["is_excluded_final"])
review_excluded = subset_review("excluded", final_review["is_excluded_final"])

dataset_variant_counts = pd.DataFrame([
    {"dataset_variant": "all_valid", "count": len(review_all_valid)},
    {"dataset_variant": "primary_training", "count": len(review_primary)},
    {"dataset_variant": "core_clean", "count": len(review_core_clean)},
    {"dataset_variant": "relation_aware", "count": len(review_relation_aware)},
    {"dataset_variant": "contradiction_ablation", "count": len(review_contradiction)},
    {"dataset_variant": "all_none_control", "count": len(review_all_none)},
    {"dataset_variant": "excluded", "count": len(review_excluded)},
])
dataset_variant_counts["percentage_of_total"] = (dataset_variant_counts["count"] / len(final_review) * 100).round(2)
display(dataset_variant_counts)

all_valid: (8030, 84)
primary_training: (6854, 84)
core_clean: (4628, 84)
relation_aware: (6837, 84)
contradiction_ablation: (1176, 84)
all_none_control: (17, 84)
excluded: (8, 84)


,dataset_variant,count,percentage_of_total
0,all_valid,8030,99.90
1,primary_training,6854,85.27
2,core_clean,4628,57.58
3,relation_aware,6837,85.06
4,contradiction_ablation,1176,14.63
5,all_none_control,17,0.21
6,excluded,8,0.10


### 10. Kalkulasi Distribusi Data per Split dan Kategori


In [10]:
def split_count_table(df, dataset_name):
    out = (
        df.groupby(["split", "dataset_role", "policy_bucket", "relation_category"], dropna=False)
        .size()
        .reset_index(name="count")
    )
    out.insert(0, "dataset_variant", dataset_name)
    return out


split_summary = pd.concat(
    [
        split_count_table(review_all_valid, "all_valid"),
        split_count_table(review_primary, "primary_training"),
        split_count_table(review_core_clean, "core_clean"),
        split_count_table(review_relation_aware, "relation_aware"),
        split_count_table(review_contradiction, "contradiction_ablation"),
        split_count_table(review_all_none, "all_none_control"),
    ],
    ignore_index=True,
)

label_distribution_rows = []
for dataset_name, df in [
    ("all_valid", review_all_valid),
    ("primary_training", review_primary),
    ("core_clean", review_core_clean),
    ("relation_aware", review_relation_aware),
    ("contradiction_ablation", review_contradiction),
]:
    for aspect in ASPECTS:
        col = f"label_text_{aspect}"
        counts = df.groupby(["split", col], dropna=False).size().reset_index(name="count")
        counts = counts.rename(columns={col: "label"})
        counts["dataset_variant"] = dataset_name
        counts["aspect"] = aspect
        label_distribution_rows.append(counts)

label_distribution_by_split = pd.concat(label_distribution_rows, ignore_index=True)
label_distribution_by_split = label_distribution_by_split[
    ["dataset_variant", "split", "aspect", "label", "count"]
].sort_values(["dataset_variant", "split", "aspect", "label"])

display(split_summary.head(30))
display(label_distribution_by_split.head(40))

,dataset_variant,split,dataset_role,policy_bucket,relation_category,count
0,all_valid,test,all_none_control,all_none_control,both_no_aspect,3
1,all_valid,test,contradiction_ablation,contradiction_ablation,correlated_but_contradictive,176
2,all_valid,test,core_clean_training,core_clean_training,correlated_non_contradictive,694
3,all_valid,test,relation_aware_training,relation_aware_training,image_aspect_only,83
4,all_valid,test,relation_aware_training,relation_aware_training,text_aspect_only,13
5,all_valid,test,relation_aware_training,relation_aware_training,uncorrelated_different_aspects,236
6,all_valid,train,all_none_control,all_none_control,both_no_aspect,11
7,all_valid,train,contradiction_ablation,contradiction_ablation,correlated_but_contradictive,824
8,all_valid,train,core_clean_training,core_clean_training,correlated_non_contradictive,3240
9,all_valid,train,relation_aware_training,relation_aware_training,image_aspect_only,386


,dataset_variant,split,aspect,label,count
60,all_valid,test,Fasilitas,Negatif,115
61,all_valid,test,Fasilitas,Netral,60
62,all_valid,test,Fasilitas,None,796
63,all_valid,test,Fasilitas,Positif,234
36,all_valid,test,Harga,Negatif,30
37,all_valid,test,Harga,Netral,25
38,all_valid,test,Harga,None,1020
39,all_valid,test,Harga,Positif,130
0,all_valid,test,Kamar,Negatif,146
1,all_valid,test,Kamar,Netral,79


### 11. Penyiapan Dataset dan Penyimpanan ke Format CSV


In [11]:
review_output_columns = [
    "ID_Review", "Platform", "Wilayah", "Nama_Hotel", "Review_Date", "Text_Review",
    "image_count", "image_files_sample", "dataset_role", "split", "split_strata",
    "relation_category", "relation_strength", "policy_bucket",
    "recommended_fusion_strategy", "is_excluded_final",
    "primary_training_candidate", "core_clean_candidate", "relation_aware_candidate",
    "contradiction_ablation_candidate",
    "text_aspects", "image_aspects", "shared_aspects",
    "hard_contradiction_aspects", "soft_disagreement_aspects",
    "text_aspect_count", "image_aspect_count", "shared_aspect_count",
    "hard_contradiction_count", "soft_disagreement_count", "relation_overlap_ratio",
]

for aspect in ASPECTS:
    review_output_columns.extend([
        f"label_text_{aspect}",
        f"label_text_id_{aspect}",
        f"label_image_agg_{aspect}",
        f"label_image_mode_{aspect}",
        f"label_image_mode_id_{aspect}",
    ])

review_output_columns = [col for col in review_output_columns if col in final_review.columns]

image_output_columns = [
    "ID_Review", "Platform", "Wilayah", "Nama_Hotel", "Review_Date", "Text_Review",
    "Gambar_Ke", "Image_FileName", "Image_Path", "image_path_found",
    "dataset_role", "split", "relation_category", "relation_strength", "policy_bucket",
    "recommended_fusion_strategy", "is_excluded_final",
    "primary_training_candidate", "core_clean_candidate", "relation_aware_candidate",
    "contradiction_ablation_candidate",
]
for aspect in ASPECTS:
    image_output_columns.extend([f"label_image_{aspect}", f"label_image_id_{aspect}"])

image_output_columns = [col for col in image_output_columns if col in image_manifest.columns]

aspect_output_columns = [
    "ID_Review", "Platform", "Wilayah", "Nama_Hotel", "Review_Date", "Text_Review",
    "image_count", "image_files_sample", "aspect", "aspect_id",
    "text_label", "text_label_id", "image_labels", "image_label_mode", "image_label_mode_id",
    "aspect_relation_status", "has_text_aspect", "has_image_aspect",
    "is_hard_contradiction", "is_text_target_positive_aspect", "is_image_target_positive_aspect",
    "dataset_role", "split", "relation_category", "relation_strength", "policy_bucket",
    "recommended_fusion_strategy", "is_excluded_final",
    "primary_training_candidate", "core_clean_candidate", "relation_aware_candidate",
    "contradiction_ablation_candidate",
]
aspect_output_columns = [col for col in aspect_output_columns if col in aspect_long.columns]


def save_csv(df, name, columns=None):
    out = df.copy()
    if columns is not None:
        out = out[columns]
    path = OUTPUT_DIR / name
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path


saved_files = []
saved_files.append(save_csv(final_review, "final_review_dataset_all_rows.csv", review_output_columns))
saved_files.append(save_csv(review_all_valid, "final_review_dataset_all_valid.csv", review_output_columns))
saved_files.append(save_csv(review_primary, "final_review_dataset_primary_training.csv", review_output_columns))
saved_files.append(save_csv(review_core_clean, "final_review_dataset_core_clean.csv", review_output_columns))
saved_files.append(save_csv(review_relation_aware, "final_review_dataset_relation_aware.csv", review_output_columns))
saved_files.append(save_csv(review_contradiction, "final_review_dataset_contradiction_ablation.csv", review_output_columns))
saved_files.append(save_csv(review_all_none, "final_review_dataset_all_none_control.csv", review_output_columns))
saved_files.append(save_csv(review_excluded, "excluded_reviews_final.csv", review_output_columns))

saved_files.append(save_csv(image_manifest, "final_image_manifest_all_rows.csv", image_output_columns))
saved_files.append(save_csv(image_manifest.loc[~image_manifest["is_excluded_final"]], "final_image_manifest_all_valid.csv", image_output_columns))
saved_files.append(save_csv(image_manifest.loc[image_manifest["primary_training_candidate"] & ~image_manifest["is_excluded_final"]], "final_image_manifest_primary_training.csv", image_output_columns))
saved_files.append(save_csv(image_manifest.loc[image_manifest["relation_aware_candidate"] & ~image_manifest["is_excluded_final"]], "final_image_manifest_relation_aware.csv", image_output_columns))
saved_files.append(save_csv(image_manifest.loc[image_manifest["contradiction_ablation_candidate"] & ~image_manifest["is_excluded_final"]], "final_image_manifest_contradiction_ablation.csv", image_output_columns))

saved_files.append(save_csv(aspect_long, "final_aspect_long_all_rows.csv", aspect_output_columns))
saved_files.append(save_csv(aspect_long.loc[~aspect_long["is_excluded_final"]], "final_aspect_long_all_valid.csv", aspect_output_columns))
saved_files.append(save_csv(aspect_long.loc[aspect_long["primary_training_candidate"] & ~aspect_long["is_excluded_final"]], "final_aspect_long_primary_training.csv", aspect_output_columns))
saved_files.append(save_csv(aspect_long.loc[aspect_long["relation_aware_candidate"] & ~aspect_long["is_excluded_final"]], "final_aspect_long_relation_aware.csv", aspect_output_columns))
saved_files.append(save_csv(aspect_long.loc[aspect_long["contradiction_ablation_candidate"] & ~aspect_long["is_excluded_final"]], "final_aspect_long_contradiction_ablation.csv", aspect_output_columns))

saved_files.append(save_csv(split_summary, "dataset_split_summary.csv"))
saved_files.append(save_csv(label_distribution_by_split, "label_distribution_by_split.csv"))
saved_files.append(save_csv(dataset_variant_counts, "dataset_variant_counts.csv"))

print("Saved CSV files:")
for path in saved_files:
    print("-", path.name)

Saved CSV files:
- final_review_dataset_all_rows.csv
- final_review_dataset_all_valid.csv
- final_review_dataset_primary_training.csv
- final_review_dataset_core_clean.csv
- final_review_dataset_relation_aware.csv
- final_review_dataset_contradiction_ablation.csv
- final_review_dataset_all_none_control.csv
- excluded_reviews_final.csv
- final_image_manifest_all_rows.csv
- final_image_manifest_all_valid.csv
- final_image_manifest_primary_training.csv
- final_image_manifest_relation_aware.csv
- final_image_manifest_contradiction_ablation.csv
- final_aspect_long_all_rows.csv
- final_aspect_long_all_valid.csv
- final_aspect_long_primary_training.csv
- final_aspect_long_relation_aware.csv
- final_aspect_long_contradiction_ablation.csv
- dataset_split_summary.csv
- label_distribution_by_split.csv
- dataset_variant_counts.csv


### 12. Penyusunan Ringkasan Eksekusi Final Dataset Builder (JSON)


In [12]:
summary = {
    "seed": SEED,
    "label_to_id": LABEL_TO_ID,
    "aspect_to_id": ASPECT_TO_ID,
    "total_review_rows": int(len(final_review)),
    "total_image_rows": int(len(image_manifest)),
    "total_aspect_rows": int(len(aspect_long)),
    "excluded_review_rows": int(final_review["is_excluded_final"].sum()),
    "valid_review_rows": int((~final_review["is_excluded_final"]).sum()),
    "dataset_role_counts": final_review["dataset_role"].value_counts().to_dict(),
    "split_counts_all_valid": review_all_valid["split"].value_counts().to_dict(),
    "split_counts_primary_training": review_primary["split"].value_counts().to_dict(),
    "split_counts_core_clean": review_core_clean["split"].value_counts().to_dict(),
    "split_counts_relation_aware": review_relation_aware["split"].value_counts().to_dict(),
    "split_counts_contradiction_ablation": review_contradiction["split"].value_counts().to_dict(),
    "image_path_found_count": int(image_manifest["image_path_found"].sum()),
    "image_path_found_percentage": round(float(image_manifest["image_path_found"].mean() * 100), 2),
    "final_policy": {
        "primary_training": "core_clean_training + relation_aware_training + all_none_control",
        "clean_baseline": "core_clean_training only",
        "relation_aware_experiment": "core_clean_training + relation_aware_training",
        "contradiction_ablation": "kept separate from clean baseline",
        "excluded": "manual_review_or_exclude or text_artifact_issue",
        "main_text_target": "label_text_*",
        "image_auxiliary_target": "label_image_* / label_image_agg_*",
    },
}

with open(OUTPUT_DIR / "final_dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label_to_id": LABEL_TO_ID, "id_to_label": ID_TO_LABEL, "aspect_to_id": ASPECT_TO_ID}, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "seed": 42,
  "label_to_id": {
    "None": 0,
    "Negatif": 1,
    "Netral": 2,
    "Positif": 3
  },
  "aspect_to_id": {
    "Kamar": 0,
    "Kebersihan": 1,
    "Pelayanan": 2,
    "Harga": 3,
    "Lokasi": 4,
    "Fasilitas": 5,
    "Makanan": 6
  },
  "total_review_rows": 8038,
  "total_image_rows": 17385,
  "total_aspect_rows": 56266,
  "excluded_review_rows": 8,
  "valid_review_rows": 8030,
  "dataset_role_counts": {
    "core_clean_training": 4628,
    "relation_aware_training": 2209,
    "contradiction_ablation": 1176,
    "all_none_control": 17,
    "excluded": 8
  },
  "split_counts_all_valid": {
    "train": 5620,
    "test": 1205,
    "val": 1205
  },
  "split_counts_primary_training": {
    "train": 4796,
    "test": 1029,
    "val": 1029
  },
  "split_counts_core_clean": {
    "train": 3240,
    "test": 694,
    "val": 694
  },
  "split_counts_relation_aware": {
    "train": 4785,
    "test": 1026,
    "val": 1026
  },
  "split_counts_contradiction_ablation": {
    "

### 13. Pembuatan Format Laporan Markdown untuk Dokumentasi


In [13]:
def dataframe_to_markdown(df):
    table = df.copy().fillna("")
    headers = [str(col) for col in table.columns]
    rows = [[str(value) for value in row] for row in table.to_numpy()]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)


methodology_notes = f"""# Methodology Notes - Notebook 03 Final Dataset Builder

## Tujuan

Notebook 03 membangun dataset final MABSA dari hasil taxonomy Notebook 02. Dataset tidak langsung mencampur semua review, tetapi dipisahkan berdasarkan `policy_bucket` dan `dataset_role`.

## Dasar Paper

1. Pontiki et al. (2014) SemEval-2014 Task 4 mendasari format aspect-level sentiment. Karena itu target utama disimpan sebagai label per aspek.
2. Xu, Mao, & Chen (2019) MIMN mendasari penggunaan interaksi text-image-aspect dalam MABSA.
3. Zhao et al. (2023) M2DF menunjukkan noisy atau text-unrelated images dapat menurunkan performa, sehingga data perlu dipisahkan menjadi clean, relation-aware, contradiction, dan excluded artifact.

## Keputusan Dataset

- `core_clean_training`: baseline multimodal utama.
- `relation_aware_training`: digunakan bersama clean data untuk model dengan gating/relevance-aware strategy.
- `contradiction_ablation`: tidak dicampur ke baseline clean, tetapi dipakai untuk stress-test/error analysis.
- `all_none_control`: dipertahankan sebagai kontrol negatif.
- `manual_review_or_exclude` dan `text_artifact_issue=True`: dikeluarkan dari final valid dataset.

## Dataset Role Counts

{dataframe_to_markdown(dataset_variant_counts)}

## Split Overview

{dataframe_to_markdown(split_overview)}

## Label Mapping

- None = 0
- Negatif = 1
- Netral = 2
- Positif = 3

## Catatan Penting

Label utama training ABSA adalah `label_text_*`. Label image dipertahankan sebagai auxiliary visual evidence dan tidak dipakai untuk menimpa label teks. Ini mencegah kontradiksi antar modality dipalsukan menjadi satu label tunggal.
"""

with open(OUTPUT_DIR / "methodology_notes_notebook_03.md", "w", encoding="utf-8") as f:
    f.write(methodology_notes)

print("Saved methodology notes:", OUTPUT_DIR / "methodology_notes_notebook_03.md")

Saved methodology notes: /kaggle/working/mabsa_final_dataset_outputs/methodology_notes_notebook_03.md


### 14. Verifikasi Akhir Integritas Data (Critical Checks)


In [14]:
checks = []

checks.append({
    "check": "no_invalid_normalized_label_cells",
    "value": int(len(invalid_cells)),
    "expected": 0,
})

checks.append({
    "check": "review_ids_unique",
    "value": int(final_review["ID_Review"].nunique()),
    "expected": int(len(final_review)),
})

checks.append({
    "check": "all_valid_has_non_excluded_split",
    "value": int(review_all_valid["split"].isin(["train", "val", "test"]).sum()),
    "expected": int(len(review_all_valid)),
})

checks.append({
    "check": "excluded_not_in_primary_training",
    "value": int((final_review["is_excluded_final"] & final_review["primary_training_candidate"]).sum()),
    "expected": 0,
})

image_valid = image_manifest.loc[~image_manifest["is_excluded_final"]]
checks.append({
    "check": "valid_images_have_split",
    "value": int(image_valid["split"].isin(["train", "val", "test"]).sum()),
    "expected": int(len(image_valid)),
})

aspect_valid = aspect_long.loc[~aspect_long["is_excluded_final"]]
checks.append({
    "check": "aspect_rows_equal_valid_review_x_7",
    "value": int(len(aspect_valid)),
    "expected": int(len(review_all_valid) * len(ASPECTS)),
})

split_leakage = (
    image_manifest.loc[~image_manifest["is_excluded_final"]]
    .groupby("ID_Review")["split"]
    .nunique()
    .gt(1)
    .sum()
)
checks.append({
    "check": "no_review_id_split_leakage_in_images",
    "value": int(split_leakage),
    "expected": 0,
})

checks_df = pd.DataFrame(checks)
checks_df["status"] = np.where(checks_df["value"] == checks_df["expected"], "OK", "NEEDS_REVIEW")
checks_df.to_csv(OUTPUT_DIR / "final_dataset_checks.csv", index=False, encoding="utf-8-sig")

display(checks_df)

if (checks_df["status"] == "NEEDS_REVIEW").any():
    print("Ada check yang perlu ditinjau sebelum lanjut Notebook 04.")
else:
    print("Notebook 03 selesai. Dataset final siap dipakai untuk Notebook 04 baseline/modeling.")

,check,value,expected,status
0,no_invalid_normalized_label_cells,0,0,OK
1,review_ids_unique,8038,8038,OK
2,all_valid_has_non_excluded_split,8030,8030,OK
3,excluded_not_in_primary_training,0,0,OK
4,valid_images_have_split,17371,17371,OK
5,aspect_rows_equal_valid_review_x_7,56210,56210,OK
6,no_review_id_split_leakage_in_images,0,0,OK


Notebook 03 selesai. Dataset final siap dipakai untuk Notebook 04 baseline/modeling.


### 15. Ekspor dan Kompresi Hasil Akhir ke Arsip ZIP


In [15]:
zip_base = OUTPUT_DIR.parent / "mabsa_final_dataset_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("ZIP output:", zip_path)
print("\nFiles in output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

ZIP output: /kaggle/working/mabsa_final_dataset_outputs.zip

Files in output directory:
- dataset_split_summary.csv
- dataset_variant_counts.csv
- excluded_reviews_final.csv
- final_aspect_long_all_rows.csv
- final_aspect_long_all_valid.csv
- final_aspect_long_contradiction_ablation.csv
- final_aspect_long_primary_training.csv
- final_aspect_long_relation_aware.csv
- final_dataset_checks.csv
- final_dataset_summary.json
- final_image_manifest_all_rows.csv
- final_image_manifest_all_valid.csv
- final_image_manifest_contradiction_ablation.csv
- final_image_manifest_primary_training.csv
- final_image_manifest_relation_aware.csv
- final_review_dataset_all_none_control.csv
- final_review_dataset_all_rows.csv
- final_review_dataset_all_valid.csv
- final_review_dataset_contradiction_ablation.csv
- final_review_dataset_core_clean.csv
- final_review_dataset_primary_training.csv
- final_review_dataset_relation_aware.csv
- label_distribution_by_split.csv
- label_mapping.json
- methodology_notes_n